# Plotly Express Interactive Visualizations: Beginner Guide
Create modern, web-ready interactive charts with hover popups, zoom controls, column faceting, and animation timelines using Plotly Express.

### 📚 What You Will Learn in this Guide:
- **Interactive Scatter & Line (`px.scatter`, `px.line`)**: How to build responsive charts you can zoom, pan, and hover over.
- **Interactive Bars & Boxes (`px.bar`, `px.box`, `px.histogram`)**: How to explore distributions with interactive tooltip popups.
- **Faceted Subplots (`facet_col`, `facet_row`)**: How to split charts across categories automatically.
- **Animated Timeline Sliders (`animation_frame`)**: How to build interactive play/pause time series animations.

> **💡 Beginner Note**: Every single concept is isolated in its own section with:
> 1. **What is this?** (Plain English explanation)
> 2. **Why do we use it?** (Real-world intuition)
> 3. **Syntax & Parameters** (Parameter-by-parameter breakdown)
> 4. **Live Python Code** with outputs using `data/raw_transactions.csv`.

In [1]:
# Step 1: Import all necessary libraries
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# Step 2: Set clean visual defaults
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['font.size'] = 10

# Step 3: Load the transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)

# Step 4: Ensure dates and numeric values are clean
df['transaction_date'] = pd.to_datetime(df['transaction_date'], format='mixed', errors='coerce')
df['transaction_amount'] = pd.to_numeric(df['transaction_amount'], errors='coerce')
df['account_age_months'] = pd.to_numeric(df['account_age_months'], errors='coerce')
df['is_fraud'] = pd.to_numeric(df['is_fraud'], errors='coerce').fillna(0).astype(int)
df = df.dropna(subset=['transaction_amount', 'transaction_date']).reset_index(drop=True)

print(f"✅ Successfully loaded {len(df)} transactions from {csv_path}")
df.head(3)

✅ Successfully loaded 14262 transactions from data/raw_transactions.csv


## 🔹 Interactive Scatter Plot: `px.scatter()`

### 1. What is this?
`px.scatter()` generates an interactive web chart where hovering your mouse over any point displays a rich tooltip with exact details.

### 2. Why do we use it?
Plotly charts are interactive out-of-the-box: you can click-and-drag to zoom in, double-click to reset, and hover to read exact numbers.

### 3. Syntax & Parameters
```python
fig = px.scatter(df, x='age', y='amount', color='card', hover_data=['id', 'region'])
fig.show()
```

In [2]:
# 1. Create interactive scatter plot
fig = px.scatter(
    df.head(200), 
    x='account_age_months', 
    y='transaction_amount', 
    color='card_type', 
    size='transaction_amount', 
    hover_data=['transaction_id', 'region'], 
    title='Interactive Scatter Plot (Hover over dots to see transaction details!)'
)

# 2. Display the interactive chart
fig.show()

Figure(Interactive Scatter Plot (Hover over dots to see transaction details!))

## 🔹 Interactive Time Series: `px.line()`

### 1. What is this?
`px.line()` creates a zoomable, interactive line chart.

### 2. Why do we use it?
Users can zoom in on specific date ranges directly in their browser.

### 3. Syntax & Parameters
```python
fig = px.line(daily_df, x='date', y='amount', markers=True)
fig.show()
```

In [3]:
# 1. Prepare daily aggregated spend
daily_trend = df.set_index('transaction_date').resample('D')['transaction_amount'].sum().reset_index().head(30)

# 2. Build interactive line chart
fig = px.line(daily_trend, x='transaction_date', y='transaction_amount', title='30-Day Daily Spend (Interactive Line Chart)', markers=True)
fig.show()

Figure(30-Day Daily Spend (Interactive Line Chart))

## 🔹 Interactive Bar Chart: `px.bar()`

### 1. What is this?
`px.bar()` creates clean categorical bar charts with grouped (`barmode='group'`) or stacked layouts.

### 2. Why do we use it?
Great for interactive business reports where users want to see exact amounts on hover.

### 3. Syntax & Parameters
```python
fig = px.bar(df, x='region', y='amount', color='card_type', barmode='group')
```

In [4]:
reg_summary = df.groupby(['region', 'card_type'])['transaction_amount'].sum().reset_index()

fig = px.bar(reg_summary, x='region', y='transaction_amount', color='card_type', barmode='group', title='Regional Spend by Card Issuer (Plotly Bar)')
fig.show()

Figure(Regional Spend by Card Issuer (Plotly Bar))

## 🔹 Interactive Box Plots: `px.box()`

### 1. What is this?
`px.box()` creates interactive boxplots where hovering shows the exact Median, Q1, Q3, Minimum, and Maximum values.

### 2. Why do we use it?
Eliminates guesswork: users see the exact dollar amounts behind the whiskers.

### 3. Syntax & Parameters
```python
fig = px.box(df, x='card_type', y='amount', color='card_type', points='outliers')
```

In [5]:
fig = px.box(df.head(300), x='card_type', y='transaction_amount', color='card_type', points='outliers', title='Interactive Box Plot with Exact Hover Quartiles')
fig.show()

Figure(Interactive Box Plot with Exact Hover Quartiles)

## 🔹 Faceted Subplots: `facet_col` and `facet_row`

### 1. What is this?
`facet_col` splits your chart into multiple subplots across horizontal columns based on a categorical variable.

### 2. Why do we use it?
Compare groups side-by-side without writing complex subplot loops.

### 3. Syntax & Parameters
```python
fig = px.scatter(df, x='age', y='amount', facet_col='card_type')
```

In [6]:
fig = px.scatter(df.head(300), x='account_age_months', y='transaction_amount', color='is_fraud', facet_col='card_type', title='Faceted Subplots Across Card Issuers (facet_col)')
fig.show()

Figure(Faceted Subplots Across Card Issuers (facet_col))

## 🔹 Animated Timeline Slider: `animation_frame`

### 1. What is this?
`animation_frame` adds an interactive timeline play/pause slider at the bottom of the chart.

### 2. Why do we use it?
Show how trends, customer growth, or revenue changes month-by-month over time dynamically.

### 3. Syntax & Parameters
```python
fig = px.scatter(df, x='age', y='amount', animation_frame='month_col')
```

In [7]:
sample_anim = df.head(150).copy()
sample_anim['month_str'] = sample_anim['transaction_date'].dt.strftime('%Y-%m')

fig = px.scatter(
    sample_anim, 
    x='account_age_months', 
    y='transaction_amount', 
    animation_frame='month_str', 
    color='card_type', 
    size='transaction_amount', 
    range_x=[0, 120], 
    range_y=[0, 2000], 
    title='Interactive Timeline Animation (Click Play below to watch!)'
)
fig.show()

Figure(Interactive Timeline Animation (Click Play below to watch!))